### Password Cracking Simulator

Method:
1. ictionary Attack: Tests words from a wordlist + mutations, Common/known passwords 
2. Brute Force Attack: Tries every possible combination, Short passwords not in any list 

In [59]:
# Step 1: Imports & Configuration
import os, time, csv, string, itertools

# Configuration
MAX_ATTEMPTS  = 1_000_000   # Dictionary attack cap
RESULTS_FILE  = "results.csv"

# Wordlist files
WORDLIST_FILES = [
    "password-wordlist.txt",   #https://gist.github.com/cihanmehmet/68abd1a11b3477ebd30eea7ef23183b5
    "rockyou.txt",  #https://www.kaggle.com/datasets/wjburns/common-password-list-rockyoutxt/data           
    "Malaysia.txt" , #https://github.com/Daveqqq/weak-passwords-top200/blob/master/Malaysia.txt
    "malaysia_wordlist.txt"
]

# Load & merge wordlists 
combined = set()
for f in WORDLIST_FILES:
    if os.path.exists(f):
        with open(f, "r", encoding="utf-8", errors="ignore") as fh:
            words = {line.strip() for line in fh if line.strip()}
            combined.update(words)
        print(f"Loaded: {os.path.basename(f)} ({len(words):,} words)")
    else:
        print(f"Skipped (not found): {f}")

wordlist     = list(combined)
wordlist_set = set(wordlist)   
print(f"\nTotal unique words: {len(wordlist):,}")


Loaded: password-wordlist.txt (233,535 words)
Loaded: rockyou.txt (14,343,688 words)
Loaded: Malaysia.txt (200 words)
Loaded: malaysia_wordlist.txt (518 words)

Total unique words: 14,451,970


In [60]:
# Step 2: Mutation Engine
# Apply **transformations** to expand password guess pool.


def generate_mutations(word: str) -> list:
    """Generate realistic password mutations from a base word."""
    mutations = [word, word.lower(), word.upper(), word.capitalize()]

    for suffix in ["1","12","123","1234","12345","0","01",
                   "2024","2023","2022","99","00","007"]:
        mutations.append(word + suffix)
        mutations.append(word.capitalize() + suffix)

    leet = word.lower().replace("a","@").replace("e","3") \
                       .replace("i","1").replace("o","0") \
                       .replace("s","$").replace("t","7")
    mutations += [leet, leet.capitalize()]

    for sym in ["!","@","#",".","_"]:
        mutations += [word + sym, word.capitalize() + sym]

    mutations += [word+"!", word+"@123", word+"123!", "123"+word]

    seen, unique = set(), []
    for m in mutations:
        if m not in seen:
            seen.add(m)
            unique.append(m)
    return unique

# Demo
sample = generate_mutations("password")
print(f"Mutations for 'password' ({len(sample)} total):")
for m in sample[:10]:
    print(f"  → {m}")
print("  ...")


Mutations for 'password' (44 total):
  → password
  → PASSWORD
  → Password
  → password1
  → Password1
  → password12
  → Password12
  → password123
  → Password123
  → password1234
  ...


In [61]:
# Step 3: Password Strength Classifier
def classify_strength(password: str) -> str:
    has_upper  = any(c.isupper() for c in password)
    has_lower  = any(c.islower() for c in password)
    has_digit  = any(c.isdigit() for c in password)
    has_symbol = any(c in string.punctuation for c in password)
    length     = len(password)
    score      = sum([has_upper, has_lower, has_digit, has_symbol])

    if length < 6:                        return "Very Weak"
    elif score == 1 and length <= 8:      return "Very Weak"
    elif length < 8 or score <= 1:        return "Weak"
    elif length < 10 or score == 2:       return "Medium"
    elif length >= 12 and score >= 3:     return "Strong"
    else:                                 return "Medium"

# Test
test_set = ["abc","daniel","danielsahid","123456","password123",
            "Malaysia2024","Tr0ub4dor&3","x9#Lm!qZ2@kP"]
print(f"{'Password':<20} {'Strength'}")
print("-" * 35)
for p in test_set:
    print(f"  {p:<20} {classify_strength(p)}")


Password             Strength
-----------------------------------
  abc                  Very Weak
  daniel               Very Weak
  danielsahid          Weak
  123456               Very Weak
  password123          Medium
  Malaysia2024         Strong
  Tr0ub4dor&3          Medium
  x9#Lm!qZ2@kP         Strong


In [62]:
# Step 4: Attack Implementations

# ATTACK 1: Dictionary Attack 
def dictionary_attack(target_password: str, words: list, wset: set) -> dict:
    """
    Phase 1: Direct lookup — O(1) instant check against full wordlist.
    Phase 2: Mutation engine — tries ~30 variations of every word.
    """
    attempts   = 0
    start_time = time.perf_counter()

    # Phase 1 — direct match
    if target_password in wset:
        elapsed = time.perf_counter() - start_time
        return {
            "password": target_password, "strength": classify_strength(target_password),
            "cracked": True, "found_as": target_password, "attempts": 1,
            "time_seconds": round(elapsed, 4), "attempts_per_sec": 1,
            "method": "Dictionary (direct match)",
        }

    # Phase 2 — mutations
    cracked, found_word = False, None
    for word in words:
        for mutated in generate_mutations(word):
            attempts += 1
            if mutated == target_password:
                cracked, found_word = True, mutated
                break
            if attempts >= MAX_ATTEMPTS:
                break
        if cracked or attempts >= MAX_ATTEMPTS:
            break

    elapsed = time.perf_counter() - start_time
    return {
        "password": target_password, "strength": classify_strength(target_password),
        "cracked": cracked, "found_as": found_word if cracked else "N/A",
        "attempts": attempts, "time_seconds": round(elapsed, 4),
        "attempts_per_sec": round(attempts / elapsed) if elapsed > 0 else 0,
        "method": "Dictionary + Mutations",
    }


# ATTACK 2: Brute Force Attack
def brute_force_attack(target_password: str, max_length: int = 6) -> dict:
    """
    Tries EVERY possible combination of lowercase letters up to max_length.
    Will always find the password if:
      - It contains only a-z characters
      - Its length is <= max_length
    """
    charset    = string.ascii_lowercase   # a-z only (26 chars)
    attempts   = 0
    cracked    = False
    found_word = None
    start_time = time.perf_counter()

    for length in range(1, max_length + 1):
        combos = 26 ** length
        print(f"  Trying length {length} ({combos:,} combinations)...")

        for combo in itertools.product(charset, repeat=length):
            attempts += 1
            guess = "".join(combo)
            if guess == target_password:
                cracked, found_word = True, guess
                break
        if cracked:
            break

    elapsed = time.perf_counter() - start_time
    return {
        "password": target_password, "strength": classify_strength(target_password),
        "cracked": cracked, "found_as": found_word if cracked else "N/A",
        "attempts": attempts, "time_seconds": round(elapsed, 4),
        "attempts_per_sec": round(attempts / elapsed) if elapsed > 0 else 0,
        "method": f"Brute Force (a-z, max {max_length} chars)",
    }


# ATTACK 3: Combined (Dictionary first, then Brute Force)
def combined_attack(target_password: str, words: list, wset: set,
                    max_bf_length: int = 6) -> dict:
    """
    Best of both worlds:
    Step 1 → Dictionary attack (fast, handles common/mutated passwords)
    Step 2 → Brute force (catches short passwords not in any wordlist)
    """
    print(f"  [1/2] Dictionary attack...")
    result = dictionary_attack(target_password, words, wset)
    if result["cracked"]:
        return result

    print(f"  [2/2] Not found in wordlist → switching to brute force (up to {max_bf_length} chars)...")
    return brute_force_attack(target_password, max_bf_length)


print("All attack functions ready.")


All attack functions ready.


In [63]:
# Step 5: Testing & Results

test_passwords = [
    "abc",            # Very short — brute forced instantly
    "daniel",         # Short lowercase — brute forced in seconds
    "danielsahid",    # Longer — NOT cracked (too long for BF)
    "123456",         # In wordlist — cracked instantly
    "password123",    # In wordlist — cracked instantly
    "Malaysia2024",   # Not in wordlist, complex — NOT cracked
    "x9#Lm!qZ2@kP",  # Very strong — NOT cracked
]

BF_MAX_LENGTH = 6     # Brute force will try up to this many characters

all_results = []

for password in test_passwords:
    strength = classify_strength(password)
    print(f"\n{'='*58}")
    print(f"  Testing  : '{password}'")
    print(f"  Strength : {strength}")
    print(f"{'='*58}")

    result = combined_attack(password, wordlist, wordlist_set, BF_MAX_LENGTH)

    status = "✅ CRACKED ⚠️" if result["cracked"] else "❌ NOT CRACKED ✅"
    print(f"\n  Result   : {status}")
    print(f"  Method   : {result['method']}")
    if result["cracked"]:
        print(f"  Found as : {result['found_as']}")
    print(f"  Attempts : {result['attempts']:,}")
    print(f"  Time     : {result['time_seconds']} seconds")
    print(f"  Speed    : {result['attempts_per_sec']:,} attempts/sec")

    all_results.append(result)



  Testing  : 'abc'
  Strength : Very Weak
  [1/2] Dictionary attack...

  Result   : ✅ CRACKED ⚠️
  Method   : Dictionary (direct match)
  Found as : abc
  Attempts : 1
  Time     : 0.0 seconds
  Speed    : 1 attempts/sec

  Testing  : 'daniel'
  Strength : Very Weak
  [1/2] Dictionary attack...

  Result   : ✅ CRACKED ⚠️
  Method   : Dictionary (direct match)
  Found as : daniel
  Attempts : 1
  Time     : 0.0 seconds
  Speed    : 1 attempts/sec

  Testing  : 'danielsahid'
  Strength : Weak
  [1/2] Dictionary attack...


  [2/2] Not found in wordlist → switching to brute force (up to 6 chars)...
  Trying length 1 (26 combinations)...
  Trying length 2 (676 combinations)...
  Trying length 3 (17,576 combinations)...
  Trying length 4 (456,976 combinations)...
  Trying length 5 (11,881,376 combinations)...
  Trying length 6 (308,915,776 combinations)...

  Result   : ❌ NOT CRACKED ✅
  Method   : Brute Force (a-z, max 6 chars)
  Attempts : 321,272,406
  Time     : 207.0603 seconds
  Speed    : 1,551,589 attempts/sec

  Testing  : '123456'
  Strength : Very Weak
  [1/2] Dictionary attack...

  Result   : ✅ CRACKED ⚠️
  Method   : Dictionary (direct match)
  Found as : 123456
  Attempts : 1
  Time     : 0.0 seconds
  Speed    : 1 attempts/sec

  Testing  : 'password123'
  Strength : Medium
  [1/2] Dictionary attack...

  Result   : ✅ CRACKED ⚠️
  Method   : Dictionary (direct match)
  Found as : password123
  Attempts : 1
  Time     : 0.0 seconds
  Speed    : 1 attempts/sec

  Testing  : 'Malaysia2024'
  St

In [64]:
# Step 6: Final Summary

print(f"\n{'='*75}")
print("  FINAL SUMMARY")
print(f"{'='*75}")
print(f"  {'Password':<18} {'Strength':<12} {'Cracked?':<10} {'Method':<25} {'Time(s)':>8}")
print(f"  {'-'*18} {'-'*12} {'-'*10} {'-'*25} {'-'*8}")

for r in all_results:
    cracked_str = "YES ⚠️ " if r["cracked"] else "NO  ✅ "
    method_short = r["method"][:24]
    print(f"  {r['password']:<18} {r['strength']:<12} {cracked_str:<10} {method_short:<25} {r['time_seconds']:>8}")



  FINAL SUMMARY
  Password           Strength     Cracked?   Method                     Time(s)
  ------------------ ------------ ---------- ------------------------- --------
  abc                Very Weak    YES ⚠️     Dictionary (direct match       0.0
  daniel             Very Weak    YES ⚠️     Dictionary (direct match       0.0
  danielsahid        Weak         NO  ✅      Brute Force (a-z, max 6   207.0603
  123456             Very Weak    YES ⚠️     Dictionary (direct match       0.0
  password123        Medium       YES ⚠️     Dictionary (direct match       0.0
  Malaysia2024       Strong       YES ⚠️     Dictionary (direct match       0.0
  x9#Lm!qZ2@kP       Strong       NO  ✅      Brute Force (a-z, max 6   175.8123


In [65]:
# Save results to CSV for later analysis

fieldnames = ["password","strength","cracked","found_as","method",
              "attempts","time_seconds","attempts_per_sec"]

with open(RESULTS_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_results:
        writer.writerow({k: r.get(k, "") for k in fieldnames})

print(f"Results saved to '{RESULTS_FILE}'")


Results saved to 'results.csv'


## Key Cyber Security

| # | Lesson |
|---|--------|
| 1 | **Dictionary attacks** instantly crack any password that exists in a wordlist |
| 2 | **Brute force** can crack ANY short/simple password, even ones not in wordlists |
| 3 | **Length** is the most powerful defence, each extra character multiplies difficulty exponentially |
| 4 | A password like `danielsahid` looks complex but is crackable given enough time and computing power |
| 5 | **True security** = long + complex + unique + MFA |
| 6 | Use a **passphrase**: `PurpleTiger$RunsFast!` - long, memorable, and extremely hard to crack |
| 7 | Use a **Password Manager** - never reuse passwords across sites |
| 8 | Even a perfect password can't protect you if the **server is breached** - enable MFA always |